# When channel effectiveness drifts, the *average* is the wrong answer

This notebook compares three Marketing Mix Models on a synthetic DGP whose channel coefficients $\beta_{c,t}$ drift over time:

1. **Robyn** — Ridge + Nevergrad. Wrong and confident (a point estimate, no uncertainty).
2. **PyMC-Marketing style** — NUTS posterior over a *fixed* $\beta_c$. Wrong but honest — wide posteriors and structured residuals signal the misspecification.
3. **DLM** — Kalman filter + RTS smoother on a random-walk-with-drift state. Correctly specified.

Fit metrics (RMSE, MAPE) are not the point. The point is the *budget allocation at $t=T$* — the forward-looking decision that depends on current, not historical-average, channel effectiveness.

In [ ]:
from dataclasses import replace

import matplotlib.pyplot as plt
import numpy as np

from dgp.config import DGPConfig
from dgp.generator import generate_dataset
from models.dlm_model import DLMModel
from models.pymc_model import PyMCModel
from models.robyn_model import RobynModel
from simulation.monte_carlo import run_monte_carlo, summarize_allocation_error
from simulation.runner import _default_dlm_kwargs
from visualization.allocation_plot import (
    plot_allocation_error_distribution,
    plot_allocation_shares,
)
from visualization.coef_plot import plot_coefficient_trajectories
from visualization.residual_plot import plot_fit_and_residuals

## The DGP

Three channels over 156 weeks: **TV** (ROI declining), **Digital** (ROI rising), **Search** (stable). Adstock is geometric, saturation is Hill; all three models receive the *same pre-transformed features* so the only question left is whether coefficient dynamics are modelled.

$$y_t = \alpha_t + \sum_c \beta_{c,t} \, a_{c,t} + \varepsilon_t$$

In [ ]:
config = DGPConfig()
data = generate_dataset(replace(config, seed=0))
X = data.feature_matrix()
y = data.y
channel_names = data.channel_names
print(f"{data.config.n_weeks} weeks, {len(channel_names)} channels: {channel_names}")

## Fit all three models on a single realization

In [ ]:
robyn = RobynModel(seed=0, nevergrad_budget=40).fit(X, y)
pymc = PyMCModel(seed=0, draws=1000, tune=1000, chains=2, target_accept=0.9).fit(X, y)
dlm = DLMModel(**_default_dlm_kwargs(config)).fit(X, y)

### Chart 1 — Coefficient recovery

True $\beta_{c,t}$ in black; each model's estimate overlaid. Robyn and PyMC collapse the whole history to a single horizontal line; the DLM's smoothed trajectory tracks the drift.

In [ ]:
fig = plot_coefficient_trajectories(
    data.beta_matrix(),
    {"robyn": robyn.beta_at_T(), "pymc": pymc.beta_at_T(), "dlm": dlm.smoothed_states[:, 1:]},
    channel_names,
)
plt.show()

### Chart 2 — Fit vs residual structure

Per-model fitted series + residuals. The misspecification of a time-invariant $\beta$ leaks into the residuals as structure — trend and seasonality the model cannot otherwise absorb. The DLM's residuals should look like white noise.

In [ ]:
fig = plot_fit_and_residuals(
    y,
    {"robyn": robyn.predict(X), "pymc": pymc.predict(X), "dlm": dlm.fitted_values()},
)
plt.show()

## Monte Carlo: distribution of allocation error across seeds

One seed is anecdotal. The decision-quality claim is about the *distribution* of outcomes — so we sweep seeds in parallel and compute the L1-on-shares allocation error for each fitter against the ground-truth optimum.

In [ ]:
# Takes ~1-2 minutes; drop --n-seeds for a quicker pass.
results = run_monte_carlo(seeds=list(range(30)), config=config, total_budget=500.0)
errors = summarize_allocation_error(results)
for name, arr in errors.items():
    print(f"{name:>6}: mean={arr.mean():.3f}  std={arr.std():.3f}  median={np.median(arr):.3f}")

### Chart 3 — Implied budget vs truth at $t=T$

In [ ]:
fig = plot_allocation_shares(results)
plt.show()

In [ ]:
fig = plot_allocation_error_distribution(results)
plt.show()

## Punchline

The three models answer different questions:

- **Robyn**: *what was average TV ROI over three years?*
- **PyMC**: *what is our posterior over a fixed TV ROI parameter?*
- **DLM**: *what is TV ROI right now, given it has been evolving?*

Only the third question is decision-relevant for forward-looking budget planning. A model that fits history beautifully but integrates the coefficient over time still prescribes the *average* allocation — which is the wrong allocation when the world has drifted.